In [1]:
import os
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

load_dotenv()

# Embeddingsの設定
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

# 既存のChromaDBを読み込む
vectorstore = Chroma(
    persist_directory="/app/data/chroma_db",
    embedding_function=embeddings,
    collection_name="diabetes_rag"
)

print(f"ChromaDB読み込み完了: {vectorstore._collection.count()}チャンク")

# LLMの設定
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

# プロンプトテンプレート
prompt_template = """
あなたは医薬品の専門家です。
以下の医薬品添付文書およびガイドラインの情報をもとに質問に回答してください。

提供された情報に含まれていない内容については「提供された文書には記載がありません」と答えてください。
提供された文書に記載がない内容は、一般的な知識であっても回答しないでください。
回答の最後に必ず参照した文書名を記載してください。

【参照情報】
{context}

【質問】
{question}

【回答】
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# RAGチェーンの構築
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print("RAGチェーン構築完了")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


ChromaDB読み込み完了: 516チャンク
RAGチェーン構築完了


In [2]:
# ガイドラインのチャンクIDを取得して削除
guideline_data = vectorstore.get(
    where={"doc_type": "guideline"}
)

ids_to_delete = guideline_data["ids"]
print(f"削除対象チャンク数: {len(ids_to_delete)}")

if ids_to_delete:
    vectorstore.delete(ids=ids_to_delete)
    print("削除完了")

print(f"削除後の総チャンク数: {vectorstore._collection.count()}")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


削除対象チャンク数: 183


Failed to send telemetry event CollectionDeleteEvent: capture() takes 1 positional argument but 3 were given


削除完了
削除後の総チャンク数: 333


In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 新しいガイドラインを読み込む
new_guideline_path = Path("/app/data/raw/guidelines/diabetes_manual_2025.pdf")
loader = PyMuPDFLoader(str(new_guideline_path))
pages = loader.load()

# メタデータを付与
for page in pages:
    page.metadata["doc_type"] = "guideline"
    page.metadata["file_name"] = new_guideline_path.name
    page.metadata["drug_name"] = new_guideline_path.stem

print(f"読み込みページ数: {len(pages)}")

# チャンク分割
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "、", ""]
)
new_chunks = splitter.split_documents(pages)
print(f"チャンク数: {len(new_chunks)}")

# ChromaDBに追加
vectorstore.add_documents(new_chunks)
print(f"追加後の総チャンク数: {vectorstore._collection.count()}")

読み込みページ数: 19
チャンク数: 145
追加後の総チャンク数: 478


In [4]:
eval_questions = [
    {
        "id": "Q01",
        "category": "禁忌",
        "question": "メトホルミンの禁忌を教えてください",
        "relevant_docs": ["metformin.pdf"],
        "relevant_keywords": [
            "乳酸アシドーシス", "重度の腎機能障害", "eGFR", "30ml/min/1.73m2未満",
            "重度の肝機能障害", "低酸素血症", "脱水", "重症ケトーシス", "糖尿病性昏睡",
            "1型糖尿病", "重症感染症", "手術", "妊娠", "過敏症", "アルコール"
        ]
    },
    {
        "id": "Q02",
        "category": "用法用量",
        "question": "グリメピリドの用法・用量を教えてください",
        "relevant_docs": ["glimepiride.pdf"],
        "relevant_keywords": [
            "0.5mg", "1mg", "1日1～2回", "朝", "食前", "食後",
            "維持量", "1～4mg", "最高投与量", "6mg"
        ]
    },
    {
        "id": "Q03",
        "category": "副作用",
        "question": "エンパグリフロジンの主な副作用を教えてください",
        "relevant_docs": ["empagliflozin.pdf"],
        "relevant_keywords": [
            "低血糖", "脱水", "ケトアシドーシス", "腎盂腎炎",
            "尿路感染症", "膀胱炎", "頻尿", "多尿", "口渇", "体重減少"
        ]
    },
    {
        "id": "Q04",
        "category": "禁忌",
        "question": "オゼンピックの禁忌を教えてください",
        "relevant_docs": ["semaglutide.pdf"],
        "relevant_keywords": [
            "過敏症", "糖尿病性ケトアシドーシス", "糖尿病性昏睡",
            "1型糖尿病", "重症感染症", "手術"
        ]
    },
    {
        "id": "Q05",
        "category": "薬剤比較",
        "question": "腎機能が低下している患者に使用できる糖尿病薬は何ですか",
        "relevant_docs": ["linagliptin.pdf", "empagliflozin.pdf"],
        "relevant_keywords": [
            "腎機能障害", "eGFR", "リナグリプチン", "用量調節不要",
            "エンパグリフロジン", "20mL/min/1.73m2"
        ]
    },
    {
        "id": "Q06",
        "category": "薬剤比較",
        "question": "DPP-4阻害薬のジャヌビアとトラゼンタの腎機能低下時の使い分けを教えてください",
        "relevant_docs": ["sitagliptin.pdf", "linagliptin.pdf"],
        "relevant_keywords": [
            "シタグリプチン", "50mg", "12.5mg", "CrCl", "30未満",
            "リナグリプチン", "5mg", "用量調節"
        ]
    },
    {
        "id": "Q07",
        "category": "薬剤比較",
        "question": "インスリン グラルギンとインスリン アスパルトの違いを教えてください",
        "relevant_docs": ["insulin_glargine.pdf", "insulin_aspart.pdf"],
        "relevant_keywords": [
            "持効型", "超速効型", "作用発現時間", "10〜20分"
        ]
    },
    {
        "id": "Q08",
        "category": "ガイドライン",
        "question": "糖尿病の血糖コントロール目標を教えてください",
        "relevant_docs": ["diabetes_manual_2025.pdf"],
        "relevant_keywords": [
            "HbA1c", "7.0%未満", "グリコアルブミン", "20%未満",
            "6.0%", "8.0%"
        ]
    },
    {
        "id": "Q09",
        "category": "ガイドライン",
        "question": "2型糖尿病の薬物療法の第一選択薬は何ですか",
        "relevant_docs": ["diabetes_manual_2025.pdf"],
        "relevant_keywords": [
            "ビグアナイド", "単剤", "1型糖尿病", "インスリン"
        ]
    },
    {
        "id": "Q10",
        "category": "横断検索",
        "question": "心血管疾患を合併した2型糖尿病患者に推奨される薬剤は何ですか",
        "relevant_docs": ["empagliflozin.pdf", "semaglutide.pdf", "diabetes_manual_2025.pdf"],
        "relevant_keywords": [
            "SGLT2阻害薬", "エンパグリフロジン", "カナグリフロジン",
            "ダパグリフロジン", "1剤上乗せ"
        ]
    },
]

print(f"評価用質問数: {len(eval_questions)}問")
for q in eval_questions:
    print(f"  {q['id']} [{q['category']}] {q['question']}")
    print(f"       キーワード数: {len(q['relevant_keywords'])}個")

評価用質問数: 10問
  Q01 [禁忌] メトホルミンの禁忌を教えてください
       キーワード数: 15個
  Q02 [用法用量] グリメピリドの用法・用量を教えてください
       キーワード数: 10個
  Q03 [副作用] エンパグリフロジンの主な副作用を教えてください
       キーワード数: 10個
  Q04 [禁忌] オゼンピックの禁忌を教えてください
       キーワード数: 6個
  Q05 [薬剤比較] 腎機能が低下している患者に使用できる糖尿病薬は何ですか
       キーワード数: 6個
  Q06 [薬剤比較] DPP-4阻害薬のジャヌビアとトラゼンタの腎機能低下時の使い分けを教えてください
       キーワード数: 8個
  Q07 [薬剤比較] インスリン グラルギンとインスリン アスパルトの違いを教えてください
       キーワード数: 4個
  Q08 [ガイドライン] 糖尿病の血糖コントロール目標を教えてください
       キーワード数: 6個
  Q09 [ガイドライン] 2型糖尿病の薬物療法の第一選択薬は何ですか
       キーワード数: 4個
  Q10 [横断検索] 心血管疾患を合併した2型糖尿病患者に推奨される薬剤は何ですか
       キーワード数: 5個


In [5]:
def evaluate_response(question_data: dict, rag_chain) -> dict:
    """
    1問分の評価を実行して結果を返す
    """
    question = question_data["question"]
    relevant_keywords = question_data["relevant_keywords"]
    
    # RAGチェーンに質問
    result = rag_chain.invoke({"query": question})
    answer = result["result"]
    source_docs = result["source_documents"]
    
    # ① キーワードスコアの計算
    matched_keywords = [
        kw for kw in relevant_keywords
        if kw in answer
    ]
    keyword_score = len(matched_keywords) / len(relevant_keywords)
    
    # ② 参照ドキュメントの適合率・再現率の計算
    retrieved_files = list(set([
        doc.metadata.get("file_name") for doc in source_docs
    ]))
    relevant_docs = question_data["relevant_docs"]
    
    # 適合率：取得したドキュメントのうち正解ドキュメントの割合
    precision = len([
        f for f in retrieved_files if f in relevant_docs
    ]) / len(retrieved_files) if retrieved_files else 0
    
    # 再現率：正解ドキュメントのうち実際に取得できた割合
    recall = len([
        f for f in retrieved_files if f in relevant_docs
    ]) / len(relevant_docs) if relevant_docs else 0
    
    # F1スコア
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0)
    
    return {
        "id": question_data["id"],
        "category": question_data["category"],
        "question": question,
        "answer": answer,
        "keyword_score": round(keyword_score, 3),
        "matched_keywords": matched_keywords,
        "unmatched_keywords": [
            kw for kw in relevant_keywords
            if kw not in matched_keywords
        ],
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "retrieved_files": retrieved_files,
        "relevant_docs": relevant_docs,
    }

print("評価関数の定義完了")

評価関数の定義完了


In [6]:
# 全問評価を実行
print("評価開始...\n")
results = []

for q in eval_questions:
    print(f"評価中: {q['id']} {q['question'][:30]}...")
    result = evaluate_response(q, rag_chain)
    results.append(result)

print("\n評価完了")
print(f"評価問題数: {len(results)}問")

評価開始...

評価中: Q01 メトホルミンの禁忌を教えてください...


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


評価中: Q02 グリメピリドの用法・用量を教えてください...
評価中: Q03 エンパグリフロジンの主な副作用を教えてください...
評価中: Q04 オゼンピックの禁忌を教えてください...
評価中: Q05 腎機能が低下している患者に使用できる糖尿病薬は何ですか...
評価中: Q06 DPP-4阻害薬のジャヌビアとトラゼンタの腎機能低下時の使い...
評価中: Q07 インスリン グラルギンとインスリン アスパルトの違いを教えて...
評価中: Q08 糖尿病の血糖コントロール目標を教えてください...
評価中: Q09 2型糖尿病の薬物療法の第一選択薬は何ですか...
評価中: Q10 心血管疾患を合併した2型糖尿病患者に推奨される薬剤は何ですか...

評価完了
評価問題数: 10問


In [7]:
# 結果をDataFrameに変換
df = pd.DataFrame([{
    "ID": r["id"],
    "カテゴリ": r["category"],
    "質問": r["question"][:25] + "...",
    "キーワードスコア": r["keyword_score"],
    "適合率": r["precision"],
    "再現率": r["recall"],
    "F1": r["f1"],
    "取得ファイル": ", ".join(r["retrieved_files"]),
} for r in results])

# 全体の平均スコア
print("=== 評価結果 ===\n")
print(df.to_string(index=False))

print("\n=== 平均スコア ===")
print(f"キーワードスコア: {df['キーワードスコア'].mean():.3f}")
print(f"適合率:           {df['適合率'].mean():.3f}")
print(f"再現率:           {df['再現率'].mean():.3f}")
print(f"F1スコア:         {df['F1'].mean():.3f}")

print("\n=== カテゴリ別平均F1スコア ===")
print(df.groupby("カテゴリ")["F1"].mean().round(3).to_string())

=== 評価結果 ===

 ID   カテゴリ                           質問  キーワードスコア   適合率   再現率    F1                                                                                            取得ファイル
Q01     禁忌         メトホルミンの禁忌を教えてください...     0.000 0.500 1.000 0.667                                                           diabetes_manual_2025.pdf, metformin.pdf
Q02   用法用量      グリメピリドの用法・用量を教えてください...     0.600 1.000 1.000 1.000                                                                                   glimepiride.pdf
Q03    副作用   エンパグリフロジンの主な副作用を教えてください...     0.000 1.000 1.000 1.000                                                                                 empagliflozin.pdf
Q04     禁忌         オゼンピックの禁忌を教えてください...     0.000 0.333 1.000 0.500                                                semaglutide.pdf, insulin_aspart.pdf, metformin.pdf
Q05   薬剤比較 腎機能が低下している患者に使用できる糖尿病薬は何で...     0.333 0.000 0.000 0.000                      metformin.pdf, diabetes_manual_2025.pdf, sitagliptin.pdf, insulin_a

In [8]:
# キーワードスコアが低い問題を詳しく確認
low_score = [r for r in results if r["keyword_score"] == 0]

for r in low_score:
    print(f"{'='*60}")
    print(f"【{r['id']}】{r['question']}")
    print(f"\n【回答】")
    print(r["answer"])
    print(f"\n【未マッチキーワード】")
    print(r["unmatched_keywords"])
    print()

【Q01】メトホルミンの禁忌を教えてください

【回答】
提供された文書には記載がありません。 

参照文書名: メトグルコ錠添付文書

【未マッチキーワード】
['乳酸アシドーシス', '重度の腎機能障害', 'eGFR', '30ml/min/1.73m2未満', '重度の肝機能障害', '低酸素血症', '脱水', '重症ケトーシス', '糖尿病性昏睡', '1型糖尿病', '重症感染症', '手術', '妊娠', '過敏症', 'アルコール']

【Q03】エンパグリフロジンの主な副作用を教えてください

【回答】
提供された文書には記載がありません。 

参照情報: 医薬品添付文書およびガイドライン

【未マッチキーワード】
['低血糖', '脱水', 'ケトアシドーシス', '腎盂腎炎', '尿路感染症', '膀胱炎', '頻尿', '多尿', '口渇', '体重減少']

【Q04】オゼンピックの禁忌を教えてください

【回答】
提供された文書にはオゼンピックの禁忌に関する具体的な記載がありません。したがって、禁忌についての情報は提供できません。

参照文書名: MOS000441

【未マッチキーワード】
['過敏症', '糖尿病性ケトアシドーシス', '糖尿病性昏睡', '1型糖尿病', '重症感染症', '手術']

【Q07】インスリン グラルギンとインスリン アスパルトの違いを教えてください

【回答】
インスリン グラルギンとインスリン アスパルトの主な違いは、構造と作用機序にあります。

1. **構造**:
   - インスリン グラルギンは、ヒトインスリンのアナログであり、特にpH4の環境で低い溶解性を示すように設計されています。皮下に投与されると微細な沈殿物を形成し、緩やかに吸収されます。
   - インスリン アスパルトは、インスリンB鎖28位のプロリン残基をアスパラギン酸に置換したインスリンアナログであり、二量体形成を阻害する性質を持っています。これにより、皮下注射後に速やかに血中に移行します。

2. **作用機序**:
   - インスリン グラルギンは、持続的な血糖降下作用を示し、24時間にわたり安定した効果を持つことが期待されます。
   - インスリン アスパルトは、速やかに血糖降下作用を発現し、短時間で効果を示します

In [9]:
# eval_questionsのQ09を修正
eval_questions[8]["relevant_keywords"] = [
    "ビグアナイド", "単剤で開始", "ステップ1",
    "eGFR", "30ml/分/1.73m2", "食事療法", "運動療法"
]

# Q10を修正
eval_questions[9]["relevant_keywords"] = [
    "SGLT2阻害薬", "心血管疾患", "心不全",
    "微量アルブミン尿", "肥満", "1剤上乗せ", "ステップ2"
]

print("キーワード修正完了")
print(f"Q09: {eval_questions[8]['relevant_keywords']}")
print(f"Q10: {eval_questions[9]['relevant_keywords']}")

キーワード修正完了
Q09: ['ビグアナイド', '単剤で開始', 'ステップ1', 'eGFR', '30ml/分/1.73m2', '食事療法', '運動療法']
Q10: ['SGLT2阻害薬', '心血管疾患', '心不全', '微量アルブミン尿', '肥満', '1剤上乗せ', 'ステップ2']


In [10]:
# ガイドラインのチャンクを確認
guideline_data = vectorstore.get(
    where={"doc_type": "guideline"}
)

print(f"ガイドラインのチャンク数: {len(guideline_data['documents'])}")
print("\n--- 全チャンクを確認 ---")
for i, doc in enumerate(guideline_data["documents"]):
    if "ビグアナイド" in doc or "ステップ" in doc or "SGLT2" in doc:
        print(f"\nチャンク{i}:")
        print(doc[:300])

ガイドラインのチャンク数: 145

--- 全チャンクを確認 ---

チャンク18:
育
11 ─13（保険適用外だが，注射薬非使用者でも血糖自己測定が血糖管理の改善に有用
69），心理的
サポート，新型コロナワクチン接種，インフルエンザワクチン接種
70, 71，肺炎球菌ワクチン接種 
72，
地域の癌検診や人間ドックなどの受診状況の確認や奨励
73, 74，災害時の対応についての支援（避
難所への持参物などの提示）．
●薬物療法
▶糖尿病の経過に伴い薬物治療およびその強化が必要となることが非常に多い
75 ─77．
▶新規に経口血糖降下薬を開始する場合，薬剤添付文書などに基づいて副作用など（表参照）につ
いて説明し 同意を得る．
▶経口血糖降下薬の選択（前表 ・ 次図

チャンク19:
大血管症への影響
主な禁忌・適応外
アジア人
欧米人
イ
ン
ス
リ
ン
分
泌
非
促
進
系
ビグアナイド薬
乳酸アシドーシス（＊＊），胃腸障害，ビ
タミンB12 欠乏症（長期服用例）
78
低
→
○（日本人）
◎（中国人）
◎
乳酸アシドーシスの既
往，過度の飲酒，（＊）
SGLT2 阻害薬
尿路性器感染症，脱水，皮疹，ケトアシドー
シス，下肢切断
79，骨折
79
低
↓
○
◎
（＊）
α─グルコシダーゼ阻害薬
肝障害，胃腸障害（放屁・下痢・ 腹満・便秘）
低
→
△
（＊）
チアゾリジン薬
浮腫，心不全，骨折（女性）
80, 81， 膀胱癌
の可能性
82 ─85，黄斑浮腫
86

チャンク21:
7
糖尿病標準診療マニュアル2025
薬剤選択は血管合併症・低血糖に関するエビデンスの有無やコストなどにより判断した．
3 〜6 ヵ月ごとに患者の病態や目標値を見直す．
薬物療法はステップ1 から開始し，その先のステップではそれぞれの薬剤を上乗せする．ステップ1 の薬剤を処
方できない場合はステップ2 から開始する．
詳細は本文を参照のこと．
糖尿病の治療の流れ
インスリンの適応か（各ステップでも考慮）
＜絶対適応＞
１型糖尿病，糖尿病昏睡・ケトアシドーシス，重度の肝障害･腎
障害・感染症，妊娠
＜相対適応＞
高血糖による症状，著明な高血糖（約300 mg/dl 以上），尿ケト
ン体陽性，経口

チャンク22:
●専門医へ適宜紹介
食事・運動療法にて数ヵ月内に

In [11]:
"""キーワード修正後の再評価"""

# 全問評価を実行
print("評価開始...\n")
results = []

for q in eval_questions:
    print(f"評価中: {q['id']} {q['question'][:30]}...")
    result = evaluate_response(q, rag_chain)
    results.append(result)

print("\n評価完了")
print(f"評価問題数: {len(results)}問")

# 結果をDataFrameに変換
df = pd.DataFrame([{
    "ID": r["id"],
    "カテゴリ": r["category"],
    "質問": r["question"][:25] + "...",
    "キーワードスコア": r["keyword_score"],
    "適合率": r["precision"],
    "再現率": r["recall"],
    "F1": r["f1"],
    "取得ファイル": ", ".join(r["retrieved_files"]),
} for r in results])

# 全体の平均スコア
print("=== 評価結果 ===\n")
print(df.to_string(index=False))

print("\n=== 平均スコア ===")
print(f"キーワードスコア: {df['キーワードスコア'].mean():.3f}")
print(f"適合率:           {df['適合率'].mean():.3f}")
print(f"再現率:           {df['再現率'].mean():.3f}")
print(f"F1スコア:         {df['F1'].mean():.3f}")

print("\n=== カテゴリ別平均F1スコア ===")
print(df.groupby("カテゴリ")["F1"].mean().round(3).to_string())

評価開始...

評価中: Q01 メトホルミンの禁忌を教えてください...
評価中: Q02 グリメピリドの用法・用量を教えてください...
評価中: Q03 エンパグリフロジンの主な副作用を教えてください...
評価中: Q04 オゼンピックの禁忌を教えてください...
評価中: Q05 腎機能が低下している患者に使用できる糖尿病薬は何ですか...
評価中: Q06 DPP-4阻害薬のジャヌビアとトラゼンタの腎機能低下時の使い...
評価中: Q07 インスリン グラルギンとインスリン アスパルトの違いを教えて...
評価中: Q08 糖尿病の血糖コントロール目標を教えてください...
評価中: Q09 2型糖尿病の薬物療法の第一選択薬は何ですか...
評価中: Q10 心血管疾患を合併した2型糖尿病患者に推奨される薬剤は何ですか...

評価完了
評価問題数: 10問
=== 評価結果 ===

 ID   カテゴリ                           質問  キーワードスコア   適合率   再現率    F1                                                                                            取得ファイル
Q01     禁忌         メトホルミンの禁忌を教えてください...     0.000 0.500 1.000 0.667                                                           diabetes_manual_2025.pdf, metformin.pdf
Q02   用法用量      グリメピリドの用法・用量を教えてください...     0.600 1.000 1.000 1.000                                                                                   glimepiride.pdf
Q03    副作用   エンパグリフロジンの主な副作用を教えてください...     0.000 1.000 1.000 1.000                            

In [12]:
import json

# ベースライン結果を保存
baseline_results = {
    "strategy": "A",
    "description": "文字数ベース chunk_size=500 overlap=50",
    "avg_keyword_score": round(df["キーワードスコア"].mean(), 3),
    "avg_precision": round(df["適合率"].mean(), 3),
    "avg_recall": round(df["再現率"].mean(), 3),
    "avg_f1": round(df["F1"].mean(), 3),
    "details": results
}

output_path = Path("/app/data/baseline_results.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)

print(f"ベースライン結果を保存しました: {output_path}")

ベースライン結果を保存しました: /app/data/baseline_results.json
